# 週次「割安高質」戦略バックテスト（J-Quantsデータ版）

**作成日**: 2026-02-18

**目的**: 週次リバランスの有効性検証、損益通算を含む税金計算、MDD削減案の検証

## 1. ライブラリインポート

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（Windows）
plt.rcParams['font.sans-serif'] = ['MS Gothic', 'Yu Gothic', 'Meiryo']
plt.rcParams['axes.unicode_minus'] = False

print("ライブラリインポート完了")

ライブラリインポート完了


## 2. データ読み込み

In [2]:
# プロジェクトルート
PROJECT_ROOT = Path(r'C:\Users\yongr\claude project\workspace')

# 価格データ
price_path = PROJECT_ROOT / 'data/curated/jquants/prices/daily_quotes_all.parquet'
df_price = pd.read_parquet(price_path)
df_price['date'] = pd.to_datetime(df_price['date'])
df_price = df_price.sort_values(['code', 'date']).reset_index(drop=True)

print(f"価格データ: {len(df_price):,} 行, {df_price['code'].nunique():,} 銘柄")
print(f"期間: {df_price['date'].min().date()} ~ {df_price['date'].max().date()}")
print()

# 財務データ
fin_path = PROJECT_ROOT / 'data/curated/jquants/financials/statements_all.parquet'
df_fin = pd.read_parquet(fin_path)
df_fin['disclosed_date'] = pd.to_datetime(df_fin['disclosed_date'])
df_fin = df_fin.sort_values(['code', 'disclosed_date']).reset_index(drop=True)

print(f"財務データ: {len(df_fin):,} 行, {df_fin['code'].nunique():,} 銘柄")
print(f"期間: {df_fin['disclosed_date'].min().date()} ~ {df_fin['disclosed_date'].max().date()}")
print()

# サンプル表示
print("価格データサンプル:")
display(df_price.head(3))
print("\n財務データサンプル:")
display(df_fin.head(3))

価格データ: 10,051,531 行, 5,308 銘柄
期間: 2016-01-15 ~ 2026-02-17

財務データ: 190,873 行, 4,663 銘柄
期間: 2016-01-15 ~ 2026-01-29

価格データサンプル:


,date,code,open,high,low,close,volume
0,2016-01-15,13010,2650.0,2660.0,2630.0,2650.0,11400.0
1,2016-01-18,13010,2620.0,2630.0,2610.0,2630.0,17300.0
2,2016-01-19,13010,2630.0,2640.0,2600.0,2600.0,14000.0



財務データサンプル:


,disclosed_date,disclosed_time,code,disclosure_number,document_type,fiscal_quarter,CurPerSt,CurPerEn,CurFYSt,fiscal_year_end,...,NxFEPS,DivAnn,FDivFY,NxFDivFY,PayoutRatioAnn,ForecastNetSales,ForecastOperatingProfit,ForecastOrdinaryProfit,ForecastProfit,ForecastEarningsPerShare
0,2016-02-05,11:30,13010,20151216469794,3QFinancialStatements_Consolidated_JP,3Q,2015-04-01,2015-12-31,2015-04-01,2016-03-31,...,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-05-09,11:30,13010,20160322439537,FYFinancialStatements_Consolidated_JP,FY,2015-04-01,2016-03-31,2015-04-01,2016-03-31,...,199.94,5.0,NaN,50.0,0.292,NaN,NaN,NaN,NaN,NaN
2,2016-08-05,11:30,13010,20160701441958,1QFinancialStatements_Consolidated_JP,1Q,2016-04-01,2016-06-30,2016-04-01,2017-03-31,...,NaN,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. 財務指標計算（PBR, ROE）

In [3]:
# BPS（1株あたり純資産）があればPBR計算可能
# ROE = (net_profit / equity) * 100

# 財務データから最新の決算データを抽出（各銘柄・各日時点で最新のもの）
df_fin_clean = df_fin[[
    'disclosed_date', 'code', 'equity', 'net_profit', 'bps', 'total_assets'
]].copy()

# ROE計算
df_fin_clean['roe'] = (df_fin_clean['net_profit'] / df_fin_clean['equity']) * 100

# 異常値フィルター
df_fin_clean = df_fin_clean[
    (df_fin_clean['roe'] > -100) & 
    (df_fin_clean['roe'] < 100) &
    (df_fin_clean['bps'] > 0) &
    (df_fin_clean['equity'] > 0)
]

print(f"財務データ（異常値除外後）: {len(df_fin_clean):,} 行")
print()
print("ROE統計:")
print(df_fin_clean['roe'].describe())
print()
print("BPS統計:")
print(df_fin_clean['bps'].describe())

財務データ（異常値除外後）: 72,933 行

ROE統計:
count    72933.000000
mean         4.371089
std         11.999795
min        -99.834163
25%          1.608767
50%          4.599201
75%          8.915017
max         96.855346
Name: roe, dtype: float64

BPS統計:
count     72933.000000
mean       4492.606816
std       26102.051633
min           0.030000
25%         484.420000
50%        1126.810000
75%        2281.420000
max      891032.000000
Name: bps, dtype: float64


## 4. 週次リバランス用のデータ準備

In [4]:
# リバランス日リスト生成（毎週金曜日）
start_date = pd.Timestamp('2017-01-01')  # 2016年は財務データ不足の可能性
end_date = df_price['date'].max()

# 全営業日
trading_days = pd.Series(df_price['date'].unique()).sort_values().reset_index(drop=True)
trading_days = trading_days[(trading_days >= start_date) & (trading_days <= end_date)]

# 毎週金曜日（曜日4）を抽出
rebalance_dates = trading_days[trading_days.dt.dayofweek == 4]

# 金曜が営業日でない場合は木曜、水曜と遡る
# 簡易版：週の最終営業日を使用
trading_days_df = pd.DataFrame({'date': trading_days})
trading_days_df['year'] = trading_days_df['date'].dt.year
trading_days_df['week'] = trading_days_df['date'].dt.isocalendar().week
rebalance_dates = trading_days_df.groupby(['year', 'week'])['date'].max().values

rebalance_dates = pd.Series(rebalance_dates).sort_values().reset_index(drop=True)

print(f"リバランス日数: {len(rebalance_dates)}")
print(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}")
print(f"頻度: 週次（年間約{len(rebalance_dates) / ((end_date - start_date).days / 365.25):.1f}回）")
print()
print("最初の10回のリバランス日:")
print(rebalance_dates.head(10).apply(lambda x: x.date()).tolist())

リバランス日数: 472
期間: 2017-01-06 ~ 2026-02-17
頻度: 週次（年間約51.7回）

最初の10回のリバランス日:
[datetime.date(2017, 1, 6), datetime.date(2017, 1, 13), datetime.date(2017, 1, 20), datetime.date(2017, 1, 27), datetime.date(2017, 2, 3), datetime.date(2017, 2, 10), datetime.date(2017, 2, 17), datetime.date(2017, 2, 24), datetime.date(2017, 3, 3), datetime.date(2017, 3, 10)]


## 5. 銘柄スクリーニング関数

In [5]:
def screen_stocks(rebalance_date, df_price, df_fin, n_stocks=20):
    """
    銘柄スクリーニング（割安高質戦略）
    
    Args:
        rebalance_date: リバランス日
        df_price: 価格データ
        df_fin: 財務データ
        n_stocks: 選定銘柄数
    
    Returns:
        選定銘柄のDataFrame（code, close, pbr, roe）
    """
    # その日の終値
    prices = df_price[df_price['date'] == rebalance_date][['code', 'close']].copy()
    
    # その日時点で利用可能な最新財務データ
    # 未来参照禁止: disclosed_date <= rebalance_date
    available_fin = df_fin[df_fin['disclosed_date'] <= rebalance_date].copy()
    
    # 各銘柄の最新データ
    latest_fin = available_fin.sort_values('disclosed_date').groupby('code').tail(1)
    
    # 価格と財務データをマージ
    merged = prices.merge(latest_fin[['code', 'bps', 'roe']], on='code', how='inner')
    
    # PBR計算
    merged['pbr'] = merged['close'] / merged['bps']
    
    # 異常値除外
    merged = merged[
        (merged['pbr'] > 0) & (merged['pbr'] < 50) &
        (merged['roe'] > -100) & (merged['roe'] < 100)
    ]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()  # データ不足
    
    # 四分位計算
    merged['pbr_q'] = pd.qcut(merged['pbr'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
    merged['roe_q'] = pd.qcut(merged['roe'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
    
    # 割安高質: PBR Q1 & ROE Q4
    candidates = merged[(merged['pbr_q'] == 'Q1') & (merged['roe_q'] == 'Q4')]
    
    if len(candidates) < n_stocks:
        # 候補不足の場合は PBR最小順で補完
        candidates = merged.nsmallest(n_stocks, 'pbr')
    
    # PBR最小順でn_stocks選定
    selected = candidates.nsmallest(n_stocks, 'pbr')[['code', 'close', 'pbr', 'roe']]
    
    return selected.reset_index(drop=True)

# テスト
test_date = rebalance_dates.iloc[0]
test_stocks = screen_stocks(test_date, df_price, df_fin_clean, n_stocks=20)
print(f"テスト（{test_date.date()}）: {len(test_stocks)} 銘柄選定")
display(test_stocks.head())

テスト（2017-01-06）: 20 銘柄選定


,code,close,pbr,roe
0,34750,132.2,0.076344,31.036384
1,48160,420.0,0.126294,11.340842
2,35370,674.3,0.135449,12.779929
3,75320,220.3,0.150446,10.197631
4,32310,399.0,0.171921,10.337680


## 6. バックテストループ（損益通算あり）

In [6]:
# パラメータ
INITIAL_CASH = 10_000_000  # 初期資本
N_STOCKS = 20  # 保有銘柄数
TAX_RATE = 0.20315  # 税率
UNIT = 100  # 単元株数

# 初期化
cash = INITIAL_CASH
portfolio = {}  # {code: {'shares': int, 'buy_price': float}}
annual_realized_pnl = 0  # 年内損益通算用
current_year = None

# 結果記録
results = []

print("バックテスト開始...")
print(f"初期資本: {INITIAL_CASH:,}円")
print(f"リバランス回数: {len(rebalance_dates)}")
print()

for i, rebalance_date in enumerate(rebalance_dates):
    # 進捗表示
    if i % 50 == 0:
        print(f"進捗: {i}/{len(rebalance_dates)} ({i/len(rebalance_dates)*100:.1f}%)")
    
    # 年の切り替わり判定
    if current_year != rebalance_date.year:
        # 年末税金処理
        if current_year is not None and annual_realized_pnl > 0:
            tax = annual_realized_pnl * TAX_RATE
            cash -= tax
            # print(f"年末税金: {tax:,.0f}円（{current_year}年）")
        
        # リセット
        annual_realized_pnl = 0
        current_year = rebalance_date.year
    
    # 既存ポートフォリオの売却
    sell_value = 0
    for code, position in portfolio.items():
        # 売却価格取得
        sell_price_data = df_price[
            (df_price['code'] == code) & 
            (df_price['date'] == rebalance_date)
        ]['close']
        
        if len(sell_price_data) > 0:
            sell_price = sell_price_data.iloc[0]
            sell_amount = position['shares'] * sell_price
            sell_value += sell_amount
            
            # 損益計算
            pnl = (sell_price - position['buy_price']) * position['shares']
            annual_realized_pnl += pnl
    
    cash += sell_value
    portfolio = {}
    
    # 新規銘柄選定
    selected = screen_stocks(rebalance_date, df_price, df_fin_clean, N_STOCKS)
    
    if len(selected) == 0:
        # 選定失敗→現金保持
        continue
    
    # 購入
    target_per_stock = cash / len(selected)
    total_invested = 0
    
    for _, row in selected.iterrows():
        code = row['code']
        price = row['close']
        
        # 100株単位で購入
        shares = int(target_per_stock / (price * UNIT)) * UNIT
        
        if shares > 0:
            invest_amount = shares * price
            total_invested += invest_amount
            
            portfolio[code] = {
                'shares': shares,
                'buy_price': price
            }
    
    cash -= total_invested
    
    # 記録
    results.append({
        'date': rebalance_date,
        'cash': cash,
        'n_stocks': len(portfolio),
        'invested': total_invested,
        'annual_pnl': annual_realized_pnl
    })

# 最終年末税金処理
if annual_realized_pnl > 0:
    tax = annual_realized_pnl * TAX_RATE
    cash -= tax
    print(f"最終年末税金: {tax:,.0f}円")

print("\nバックテスト完了")
print(f"最終現金: {cash:,.0f}円")
print(f"最終保有銘柄数: {len(portfolio)}")

バックテスト開始...
初期資本: 10,000,000円
リバランス回数: 472

進捗: 0/472 (0.0%)


KeyboardInterrupt: 

## 7. 日次パフォーマンス計算

In [ ]:
# 日次のポートフォリオ価値を計算
# 簡易版：リバランス日のスナップショットのみ記録

df_results = pd.DataFrame(results)
df_results['total_value'] = df_results['cash'] + df_results['invested']
df_results['return'] = df_results['total_value'].pct_change()
df_results['cumulative_return'] = (1 + df_results['return']).cumprod() - 1

# 税引後リターン計算（簡易版）
# 各年末の税金を考慮
# （詳細版は全リバランス時の税金を逐次反映済み）

print("パフォーマンスサマリ:")
print(f"期間: {df_results['date'].iloc[0].date()} ~ {df_results['date'].iloc[-1].date()}")
print(f"初期資本: {INITIAL_CASH:,}円")
print(f"最終資産: {df_results['total_value'].iloc[-1]:,.0f}円")
print(f"総リターン: {df_results['cumulative_return'].iloc[-1]*100:.2f}%")
print()

# 年率換算
years = (df_results['date'].iloc[-1] - df_results['date'].iloc[0]).days / 365.25
total_return = df_results['cumulative_return'].iloc[-1]
annual_return = (1 + total_return) ** (1 / years) - 1

print(f"年率リターン: {annual_return*100:.2f}%")
print(f"年率ボラティリティ: {df_results['return'].std() * np.sqrt(52):.2%}")

# MDD計算
df_results['peak'] = df_results['total_value'].cummax()
df_results['drawdown'] = (df_results['total_value'] - df_results['peak']) / df_results['peak']
mdd = df_results['drawdown'].min()

print(f"最大ドローダウン: {mdd:.2%}")

# Sharpe ratio（リスクフリーレート0%仮定）
sharpe = df_results['return'].mean() / df_results['return'].std() * np.sqrt(52)
print(f"シャープレシオ: {sharpe:.2f}")

# Calmar ratio
calmar = annual_return / abs(mdd)
print(f"カルマー比: {calmar:.2f}")

## 8. ビジュアライゼーション

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 累積リターン
axes[0].plot(df_results['date'], df_results['cumulative_return'] * 100, label='週次戦略', linewidth=2)
axes[0].set_title('累積リターン推移', fontsize=14, fontweight='bold')
axes[0].set_ylabel('累積リターン (%)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# ドローダウン
axes[1].fill_between(df_results['date'], df_results['drawdown'] * 100, 0, alpha=0.3, color='red')
axes[1].set_title('ドローダウン推移', fontsize=14, fontweight='bold')
axes[1].set_ylabel('ドローダウン (%)', fontsize=12)
axes[1].grid(True, alpha=0.3)

# 資産推移
axes[2].plot(df_results['date'], df_results['total_value'] / 1e6, label='総資産', linewidth=2)
axes[2].set_title('総資産推移', fontsize=14, fontweight='bold')
axes[2].set_ylabel('総資産 (百万円)', fontsize=12)
axes[2].set_xlabel('日付', fontsize=12)
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

## 9. 年次パフォーマンス詳細

In [ ]:
# 年次リターンを計算
df_results['year'] = df_results['date'].dt.year

annual_summary = []
for year in df_results['year'].unique():
    year_data = df_results[df_results['year'] == year]
    
    if len(year_data) > 1:
        year_return = (year_data['total_value'].iloc[-1] / year_data['total_value'].iloc[0]) - 1
        
        annual_summary.append({
            'year': year,
            'return': year_return,
            'start_value': year_data['total_value'].iloc[0],
            'end_value': year_data['total_value'].iloc[-1],
            'n_rebalances': len(year_data)
        })

df_annual = pd.DataFrame(annual_summary)
print("年次パフォーマンス:")
display(df_annual)

## 10. MDD削減・リターン向上案の検証（TODO）

In [ ]:
# TODO: 以下の改善案を実装・検証
# 1. ボラティリティ調整
# 2. モメンタムフィルター
# 3. ストップロス
# 4. セクター分散

print("改善案の検証は次のセクションで実装予定")